# Exercício 03 — Custom Scikit-learn API
### Mastering Machine Learning Advanced · ML Engineer Track
### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

## Por que isso importa para um ML Engineer?

Em projetos reais, os transformadores padrão do sklearn raramente são suficientes. Você vai precisar criar:

- Transformadores com lógica de negócio customizada
- Estimadores que encapsulam modelos proprietários
- Pipelines que funcionam com `GridSearchCV` e `cross_val_score`

Dominar a API do sklearn (`fit`, `transform`, `fit_transform`, `BaseEstimator`, `TransformerMixin`) é essencial para qualquer ML Engineer que trabalha em produção.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# carregando o dataset Telecom Churn
url = 'https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['customerID'])
print(f'Dataset carregado: {df.shape}')

---
## Exercício 3.1 — TelcoFeatureEngineer

Crie um transformer customizado que aplica as regras de Feature Engineering específicas do domínio Telecom (as que vimos na Aula 1), mas encapsuladas em uma classe reutilizável compatível com Pipeline.

**Features a criar:**
- `tenure_group`: faixa de tempo como cliente (bins: 0-12, 12-24, 24-48, 48+)
- `num_servicos`: contagem de serviços adicionais contratados
- `charges_ratio`: TotalCharges / (MonthlyCharges + 1)
- `is_new_client`: 1 se tenure <= 6 meses, 0 caso contrário

In [ ]:
SERVICOS = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

class TelcoFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Transformer customizado com regras de Feature Engineering
    específicas do domínio Telecom.

    Compatível com sklearn Pipeline e GridSearchCV.
    """

    def __init__(self, criar_tenure_group=True, criar_num_servicos=True,
                 criar_charges_ratio=True, criar_is_new=True):
        self.criar_tenure_group  = criar_tenure_group
        self.criar_num_servicos  = criar_num_servicos
        self.criar_charges_ratio = criar_charges_ratio
        self.criar_is_new        = criar_is_new

    def fit(self, X, y=None):
        # SEU CÓDIGO AQUI
        # Um transformer stateless pode retornar self diretamente
        # Se precisar calcular algo do treino (ex: mediana), faça aqui
        # Obrigatório: retorne self
        pass

    def transform(self, X, y=None):
        # SEU CÓDIGO AQUI
        # 1. faça uma cópia de X (nunca modifique o original)
        # 2. crie as features conforme os flags em __init__
        # 3. retorne o DataFrame transformado
        # Dica: use check_is_fitted(self) se tiver estado do fit
        pass


# --- VALIDAÇÃO ---
fe = TelcoFeatureEngineer()
df_transformado = fe.fit_transform(df.drop(columns=['Churn']))

assert 'num_servicos' in df_transformado.columns, 'Feature num_servicos faltando'
assert 'charges_ratio' in df_transformado.columns, 'Feature charges_ratio faltando'
assert 'is_new_client' in df_transformado.columns, 'Feature is_new_client faltando'
assert df_transformado.shape[0] == df.shape[0], 'Número de linhas não deve mudar'

print(f'Shape original: {df.shape} → Transformado: {df_transformado.shape}')
print(f'Novas features: {[c for c in df_transformado.columns if c not in df.columns]}')
print('✅ Exercício 3.1 correto!')

---
## Exercício 3.2 — OutlierCapper

Crie um transformer que faz capping de outliers usando percentis calculados **no treino** e aplicados no teste. Esse é um exemplo de transformer **com estado** — o comportamento no `transform` depende do que foi aprendido no `fit`.

In [ ]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    """
    Faz capping de outliers nos percentis lower/upper calculados no treino.
    Aplica apenas em colunas numéricas.
    """

    def __init__(self, lower_pct=0.01, upper_pct=0.99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        # SEU CÓDIGO AQUI
        # calcule e armazene os limites inferior e superior
        # para cada coluna numérica usando os percentis do treino
        # armazene em self.limits_ = {coluna: (lower, upper)}
        pass

    def transform(self, X, y=None):
        # SEU CÓDIGO AQUI
        # use check_is_fitted(self, 'limits_')
        # aplique os limites aprendidos no fit
        # use pd.DataFrame.clip() ou np.clip()
        pass


# --- VALIDAÇÃO ---
from sklearn.model_selection import train_test_split

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_num = df[cols_num].dropna()
X_num_tr, X_num_te = train_test_split(X_num, test_size=0.2, random_state=42)

capper = OutlierCapper(lower_pct=0.05, upper_pct=0.95)
capper.fit(X_num_tr)
X_capped = capper.transform(X_num_te)

assert X_capped.shape == X_num_te.shape, 'Shape não deve mudar'
assert hasattr(capper, 'limits_'), 'OutlierCapper deve ter atributo limits_ após fit'

# verificando que os limites foram aplicados
for col in cols_num:
    lower, upper = capper.limits_[col]
    assert X_capped[col].max() <= upper + 1e-6, f'{col}: valor acima do limite superior'
    assert X_capped[col].min() >= lower - 1e-6, f'{col}: valor abaixo do limite inferior'

print('✅ Exercício 3.2 correto!')

---
## Exercício 3.3 — Pipeline com GridSearchCV

O maior poder da API do sklearn é que transformers customizados funcionam nativamente com `GridSearchCV`. Monte um pipeline que use suas classes customizadas e faça tuning dos hiperparâmetros — **incluindo os parâmetros do seu transformer**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = ['gender', 'Partner', 'Contract', 'PaymentMethod', 'InternetService']

X = df[cols_num + cols_cat]
y = df['Churn']

# SEU CÓDIGO AQUI
# Monte um Pipeline com pelo menos:
#   1. OutlierCapper nas features numéricas
#   2. StandardScaler nas features numéricas
#   3. OneHotEncoder nas features categóricas
#   4. RandomForestClassifier como modelo final
#
# Depois execute um GridSearchCV com pelo menos:
#   - 2 valores para n_estimators do Random Forest
#   - 2 valores para max_depth
#   - 2 valores para upper_pct do OutlierCapper
#
# Use StratifiedKFold(n_splits=3) e scoring='roc_auc'

pipeline_custom = None  # substitua pela sua implementação
grid = None             # substitua pela sua implementação

# --- VALIDAÇÃO ---
# grid.fit(X, y)
# print(f'Melhores parâmetros: {grid.best_params_}')
# print(f'Melhor AUC-ROC: {grid.best_score_:.4f}')
# assert grid.best_score_ > 0.75, 'AUC-ROC esperada > 0.75'
# print('✅ Exercício 3.3 correto!')

---
## Exercício 3.4 — Classificador Customizado

Crie um classificador que implementa a lógica de **regras de negócio** como um estimador sklearn válido. O modelo deve ser compatível com `cross_val_score` e `Pipeline`.

**Regra de negócio:** um cliente é classificado como churn se pelo menos 2 das 3 condições forem verdadeiras:
1. `tenure < 12` meses
2. `Contract == 'Month-to-month'`
3. `MonthlyCharges > mediana do treino`

In [ ]:
class RuleBasedChurnClassifier(BaseEstimator, ClassifierMixin):
    """
    Classificador baseado em regras de negócio para churn em Telecom.
    Compatível com sklearn API.
    """

    def fit(self, X, y=None):
        # SEU CÓDIGO AQUI
        # aprenda a mediana de MonthlyCharges no treino
        # armazene em self.median_monthly_charges_
        # defina self.classes_ = np.array([0, 1])
        pass

    def predict(self, X):
        # SEU CÓDIGO AQUI
        # aplique as 3 regras de negócio
        # retorne 1 se >= 2 regras forem verdadeiras, 0 caso contrário
        pass

    def predict_proba(self, X):
        # SEU CÓDIGO AQUI
        # retorne uma matriz (n_samples, 2)
        # onde a coluna 1 é a "probabilidade" de churn
        # use o número de regras verdadeiras / 3 como score
        pass


# --- VALIDAÇÃO ---
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_rb = df[['tenure', 'Contract', 'MonthlyCharges']]
y_rb = df['Churn']

clf_rules = RuleBasedChurnClassifier()
scores = cross_val_score(
    clf_rules, X_rb, y_rb,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f'Rule-based classifier — AUC-ROC: {scores.mean():.4f} ± {scores.std():.4f}')
assert scores.mean() > 0.6, 'AUC-ROC esperada > 0.60'
print('✅ Exercício 3.4 correto!')

**Reflexão:** regras de negócio como classificador têm valor real em produção? Quando usaria esse padrão?

*(Escreva sua resposta aqui)*

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)